In [1]:
!pwd

/Users/davidbird/github-work/quarto-blog


In [4]:
!head -5 src/waste_operators/load.py

"""Load raw source files into DuckDB tables. Raw files live in data/raw (not committed)."""


In [12]:
!cp ~/Desktop/load.py src/waste_operators/load.py

In [13]:
!pip install pyxlsb
!ls data/raw

2025 Waste Data Interrogator - Wastes Received (Excel) - Version 2.xlsb
2025 Waste Data Interrogator - Wastes Removed (Excel) - Version 2.xlsb
2025 Waste Summary Tables for England - Version 2


In [15]:
%run src/waste_operators/load.py


=== 2025 Waste Data Interrogator - Wastes Received (Excel) - Version 2.xlsb ===
wrote data/interim/waste_received_2025.parquet

rows: 365,819   columns: 29
distinct EPR numbers: 6,307
distinct operator names: 3,782
distinct site names: 6,023
total tonnes_received: 260,091,338

missing values (top 10 columns):
facility_london_wapg    353289
form                     20859
facility_rpa               723
facility_wpa               723
facility_district          723
facility_sub_region        723
origin_wpa                   6
recorded_origin              6
origin_region                6
post_code                    5

top 10 operators by tonnes:
operator
Thames Water Utilities Limited        31995663.0
United Utilities Water Limited        12031243.0
SUEZ Recycling and Recovery UK Ltd     6570600.0
YORKSHIRE WATER SERVICES LIMITED       5609677.0
Severn Trent Water Limited             5134238.0
EMR GROUP LIMITED                      4480434.0
BIFFA WASTE SERVICES LIMITED           3969747

In [16]:
import duckdb
duckdb.sql("SELECT operator, SUM(tonnes_received) AS tonnes FROM 'data/interim/waste_received_2025.parquet' GROUP BY 1 ORDER BY 2 DESC LIMIT 10").df()

,operator,tonnes
0,Thames Water Utilities Limited,3.199566e+07
1,United Utilities Water Limited,1.203124e+07
2,SUEZ Recycling and Recovery UK Ltd,6.570600e+06
3,YORKSHIRE WATER SERVICES LIMITED,5.609677e+06
4,Severn Trent Water Limited,5.134238e+06
5,EMR GROUP LIMITED,4.480434e+06
6,BIFFA WASTE SERVICES LIMITED,3.969747e+06
7,MICK GEORGE LIMITED,3.232961e+06
8,Anglian Water Services Limited,2.797129e+06
9,VEOLIA ES (UK) LIMITED,2.166069e+06


In [17]:
import duckdb
duckdb.sql("""
SELECT facility_type, COUNT(DISTINCT epr_number) AS permits,
       ROUND(SUM(tonnes_received)/1e6, 1) AS million_tonnes
FROM 'data/interim/waste_received_2025.parquet'
GROUP BY 1 ORDER BY 3 DESC LIMIT 10
""").df()

,facility_type,permits,million_tonnes
0,Non-Haz Waste Transfer,984,29.1
1,Physical Treatment,725,28.3
2,Anaerobic digestion,173,28.1
3,Inert LF,134,14.8
4,Physical-Chemical Treatment,163,14.2
5,Biological Treatment,148,13.3
6,Municipal Waste Incinerator,45,13.0
7,Non Hazardous LF,105,11.6
8,Metal Recycling,478,10.8
9,Recovery of Waste,84,9.6


In [18]:
duckdb.sql(r"""
WITH ops AS (SELECT DISTINCT operator FROM 'data/interim/waste_received_2025.parquet'),
n AS (
  SELECT operator,
         regexp_replace(
           regexp_replace(regexp_replace(lower(operator), '&', ' and '), '[^a-z0-9 ]', ' ', 'g'),
           '\b(limited|ltd|plc|llp)\b', ' ', 'g') AS stem
  FROM ops)
SELECT COUNT(*) AS raw_names,
       COUNT(DISTINCT trim(regexp_replace(stem, '\s+', ' ', 'g'))) AS standardised_names
FROM n
""").df()

,raw_names,standardised_names
0,3782,3780


In [19]:
f = "'data/interim/waste_received_2025.parquet'"
q = lambda sql: duckdb.sql(sql).df()

# 1. What is Thames Water's tonnage?
q(f"""SELECT ewc_chapter, facility_type, ROUND(SUM(tonnes_received)/1e6,1) AS mt
FROM {f} WHERE operator = 'Thames Water Utilities Limited'
GROUP BY 1,2 ORDER BY 3 DESC LIMIT 8""")


,ewc_chapter,facility_type,mt
0,19 - WASTE AND WATER TREATMENT WASTES,Anaerobic digestion,16.3
1,19 - WASTE AND WATER TREATMENT WASTES,Physical-Chemical Treatment,7.0
2,19 - WASTE AND WATER TREATMENT WASTES,Combustion,5.4
3,19 - WASTE AND WATER TREATMENT WASTES,Biological Treatment,2.3
4,19 - WASTE AND WATER TREATMENT WASTES,Recovery of Waste,0.9
5,16 - WASTES NOT OTHERWISE SPECIFIED IN THE LIST,Biological Treatment,0.0
6,16 - WASTES NOT OTHERWISE SPECIFIED IN THE LIST,Anaerobic digestion,0.0
7,16 - WASTES NOT OTHERWISE SPECIFIED IN THE LIST,Combustion,0.0


In [20]:
# 2. Biggest waste chapters overall
q(f"""SELECT ewc_chapter, ROUND(SUM(tonnes_received)/1e6,1) AS mt,
       COUNT(DISTINCT operator) AS operators
FROM {f} GROUP BY 1 ORDER BY 2 DESC LIMIT 10""")


,ewc_chapter,mt,operators
0,19 - WASTE AND WATER TREATMENT WASTES,100.9,1121
1,17 - CONSTRUCTION AND DEMOLITION WASTES,78.6,2110
2,20 - MUNICIPAL WASTES,58.8,1742
3,02 - AGRICULTURE AND FOOD PROCESSING WASTES,6.7,552
4,16 - WASTES NOT OTHERWISE SPECIFIED IN THE LIST,5.2,1493
5,"15 - PACKAGING, ABSORBENTS , WIPING CLOTHS ETC...",4.5,631
6,10 - THERMAL PROCESSES WASTE,1.6,225
7,13 - OIL WASTES AND WASTES OF LIQUID FUELS,0.8,237
8,18 - HUMAN AND ANIMAL HEALTH CARE WASTE,0.6,184
9,07 - ORGANIC CHEMICAL PROCESS WASTE,0.5,140


In [21]:
# 3. What does 'fate' contain?
q(f"""SELECT fate, ROUND(SUM(tonnes_received)/1e6,1) AS mt
FROM {f} GROUP BY 1 ORDER BY 2 DESC""")


,fate,mt
0,Recovery,178.9
1,Landfill,29.3
2,Incineration,23.4
3,Treatment,14.5
4,Transfer (D),13.3
5,Other Fate,0.7
6,None,0.0
7,Long term storage,0.0


In [22]:
# 4. How fragmented are the big groups? (first word of the name is a rough proxy)
q(f"""SELECT regexp_extract(lower(operator), '^[a-z0-9]+', 0) AS first_word,
       COUNT(DISTINCT operator) AS names, ROUND(SUM(tonnes_received)/1e6,1) AS mt
FROM {f} GROUP BY 1 HAVING COUNT(DISTINCT operator) > 1 ORDER BY 3 DESC LIMIT 15""")

,first_word,names,mt
0,thames,2,32.4
1,united,5,12.1
2,veolia,17,9.3
3,suez,9,9.2
4,biffa,8,6.5
5,severn,11,6.1
6,yorkshire,4,5.7
7,emr,2,4.5
8,fcc,7,3.7
9,mick,4,3.3


In [23]:
q(f"""SELECT ewc_sub_chapter, ROUND(SUM(tonnes_received)/1e6,1) AS mt,
       COUNT(DISTINCT operator) AS operators
FROM {f} WHERE ewc_chapter LIKE '19%' GROUP BY 1 ORDER BY 2 DESC LIMIT 8""")

,ewc_sub_chapter,mt,operators
0,Waste water treatment plants not otherwise spe...,52.6,133
1,Mechanical treatment of waste (for example sor...,28.3,934
2,Physico/chemical treatments of waste (includin...,3.8,171
3,Incineration or pyrolysis of waste,3.7,102
4,Anaerobic treatment of waste,3.3,79
5,Landfill leachate,2.7,58
6,Soil and groundwater remediation,2.6,62
7,Shredding of metal-containing wastes,1.3,77


In [24]:
import sys; sys.path.append("src")
from waste_operators.clean import split_legal_form

ops = duckdb.sql(f"""
SELECT operator, COUNT(DISTINCT epr_number) AS permits,
       ROUND(SUM(tonnes_received)) AS tonnes_received
FROM {f} WHERE ewc_chapter NOT LIKE '19%'
GROUP BY operator ORDER BY tonnes_received DESC
""").df()

ops[["stem", "legal_form"]] = ops["operator"].apply(lambda n: pd.Series(split_legal_form(n)))
print(len(ops), ops["stem"].nunique())
print(ops["legal_form"].value_counts(dropna=False).head())
ops.head(15)

3683 3681
legal_form
limited    2583
            750
ltd         313
plc          19
llp          17
Name: count, dtype: int64


,operator,permits,tonnes_received,stem,legal_form
0,SUEZ Recycling and Recovery UK Ltd,87,5287246.0,suez recycling and recovery uk,ltd
1,BIFFA WASTE SERVICES LIMITED,73,3257704.0,biffa waste services,limited
2,MICK GEORGE LIMITED,25,2754498.0,mick george,limited
3,EMR GROUP LIMITED,66,2242113.0,emr group,limited
4,VEOLIA ES (UK) LIMITED,69,1879516.0,veolia es uk,limited
5,ASSOCIATED WASTE MANAGEMENT LIMITED,18,1741987.0,associated waste management,limited
6,Tarmac Trading Limited,17,1617036.0,tarmac trading,limited
7,Holcim UK Limited,24,1494171.0,holcim uk,limited
8,FCC WASTE SERVICES (UK) LIMITED,48,1432389.0,fcc waste services uk,limited
9,Sewells Reservoir Construction Limited,11,1353283.0,sewells reservoir construction,limited


In [25]:
# The 750 with no legal form: what are they?
ops[ops["legal_form"] == ""].head(25)[["operator", "permits", "tonnes_received"]]

# The two stem collisions
ops[ops.duplicated("stem", keep=False)].sort_values("stem")

,operator,permits,tonnes_received,stem,legal_form
430,B T Jenkins,1,79855.0,b t jenkins,
1130,B.T. JENKINS LIMITED,1,19280.0,b t jenkins,limited
69,CHURCHILL ENVIRO LIMITED,3,380001.0,churchill enviro,limited
1620,Churchill Enviro Ltd,1,9000.0,churchill enviro,ltd


In [26]:
duckdb.sql("COPY ops TO 'data/interim/operators_in_scope.parquet' (FORMAT PARQUET)")

In [27]:
ops[ops["legal_form"] == ""].head(25)[["operator", "permits", "tonnes_received"]]

,operator,permits,tonnes_received
17,TAMWORTH BOROUGH COUNCIL,1,901654.0
51,Roland Head,1,440830.0
61,ESSEX COUNTY COUNCIL,25,398414.0
91,Keltbray Plant Ltd trading as Haulage and Envi...,1,317340.0
114,Hertfordshire County Council,17,280863.0
116,LINCOLNSHIRE COUNTY COUNCIL,19,278378.0
128,LANCASHIRE COUNTY COUNCIL,19,261013.0
137,Jay Mould,1,252225.0
138,CITY OF BRADFORD METROPOLITAN DISTRICT COUNCIL,9,250227.0
168,DURHAM COUNTY COUNCIL,6,201133.0


In [31]:
ops[ops["legal_form"] == ""][["operator", "permits", "tonnes_received", "stem"]].to_csv("~/Downloads/750records.csv", index=False)

In [30]:
ops.head()

,operator,permits,tonnes_received,stem,legal_form
0,SUEZ Recycling and Recovery UK Ltd,87,5287246.0,suez recycling and recovery uk,ltd
1,BIFFA WASTE SERVICES LIMITED,73,3257704.0,biffa waste services,limited
2,MICK GEORGE LIMITED,25,2754498.0,mick george,limited
3,EMR GROUP LIMITED,66,2242113.0,emr group,limited
4,VEOLIA ES (UK) LIMITED,69,1879516.0,veolia es uk,limited


In [32]:
q(f"""SELECT operator, site_name, facility_type, ROUND(SUM(tonnes_received)) AS tonnes
FROM {f}
WHERE operator IN ('TAMWORTH BOROUGH COUNCIL','Roland Head','Jay Mould','Pamela O''Gara')
  AND ewc_chapter NOT LIKE '19%'
GROUP BY ALL ORDER BY tonnes DESC LIMIT 10""")

,operator,site_name,facility_type,tonnes
0,TAMWORTH BOROUGH COUNCIL,Tamworth Borough Council Sandy Way,Non-Haz Waste Transfer,901654.0
1,Roland Head,Cross Croft Ind Est,Metal Recycling,440830.0
2,Jay Mould,Reading Quarry,Inert LF,252225.0
3,Pamela O'Gara,Gawsworth Quarry,Deposit of waste to land (recovery),145354.0
